# 18 · Local vision-language and staged-attribute comparators

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Optional heavyweight model comparisons. These are documented adaptations, not claimed exact FoodCHA or ConfLVLM reproductions.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Install the optional local VLM stack
Keep Colab’s torch stack. Models are downloaded only when this cell is explicitly run.

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','accelerate>=1.0,<2','huggingface-hub>=0.27,<1','safetensors>=0.4'],check=True)

## 2. Pin a selected model snapshot and load it

In [ ]:
from oncoplate.multimodal import pin_model,LocalVLM,run_vlm
FAMILY='qwen'  # Repeat with 'smol'; run one family at a time to avoid needless VRAM use.
entry=cfg['vlm'][FAMILY]
revision=pin_model(entry['model_id'],p['private']/f'{FAMILY}_model_snapshot.json',entry.get('revision'))
QUANTIZE_4BIT=False  # Enabling needs bitsandbytes and a separately recorded model condition.
model=LocalVLM(entry['model_id'],entry['family'],revision,quantize_4bit=QUANTIZE_4BIT)
print(entry['model_id'],revision)

## 3. Use development images and permitted metadata only

In [ ]:
from oncoplate.inputs import load_context
records,targets,states,rules=load_context(cfg)
subset=records[records.split.eq('validation')].head(32).copy()  # Smoke; remove .head after checking the actual GPU.
fields=list(rules);values={f:r['values'] for f,r in rules.items()}
metadata={rid:{'sources':s.get('sources',[]),'focal_item_id':s['focal_item_id']} for rid,s in states.items()}

## 4. Compare direct and staged prediction under the same inputs
Malformed outputs remain errors/abstentions; do not repair them from reference labels.

In [ ]:
for protocol_name in ['direct','staged']:
    out=p['private']/'vlm'/f'{FAMILY}_{protocol_name}_validation.jsonl'
    rows=run_vlm(model,subset,out,fields=fields,values=values,mode='multimodal',metadata=metadata,protocol=protocol_name,max_new_tokens=cfg['vlm']['max_new_tokens'])
    print(protocol_name,'records:',len(rows),'parse failures:',sum(bool(r.get('error')) for r in rows))

## 5. Expand only after checking model licence and snapshot reproducibility

In [ ]:
print('Repeat image_only, metadata_only and multimodal as separately named experiments.')
print('For the sealed test, add exact model/prompt snapshots to the analysis lock BEFORE opening test inputs.')
print('Open-ended VLM outputs need blind reference ratings; the primary finite-claim comparison does not inherit a published conformal guarantee.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
